# Variational Autoencoder (VAE) — NIDS Training

Train multiple VAE configurations on Monday benign traffic from CICIDS2017.

**Key idea:** VAE learns a probabilistic latent space of normal traffic.  
Anomalies produce higher reconstruction error + KL divergence.

In [ ]:
import sys, os

# Mount Google Drive and add the project directory to Python path
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json, datetime
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from tensorflow.keras import layers, callbacks

import joblib
import nids_utils as nu

print(f"TF version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. Load & Preprocess Data

In [ ]:
SCALER_SAVE = os.path.join(nu.PREPROCESSING_DIR, "scaler_vae.pkl")

X_train, scaler, feature_cols = nu.prepare_train_data(
    monday_path=nu.MONDAY_CSV,
    scaler_save_path=SCALER_SAVE,
)

input_dim = X_train.shape[1]
print(f"Training samples: {X_train.shape[0]}, features: {input_dim}")

## 2. VAE Model Definition

In [ ]:
class Sampling(layers.Layer):
    """Reparameterisation trick: z = mu + sigma * epsilon."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


def build_vae(input_dim, encoder_units, latent_dim, decoder_units, dropout, beta):
    """
    Build a dense VAE.

    Parameters
    ----------
    input_dim      : int
    encoder_units  : list[int]  – hidden sizes for encoder
    latent_dim     : int
    decoder_units  : list[int]  – hidden sizes for decoder
    dropout        : float
    beta           : float      – weight on KL term (beta-VAE)
    """
    # ---------- Encoder ----------
    inp = layers.Input(shape=(input_dim,))
    x = inp
    for units in encoder_units:
        x = layers.Dense(units, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)

    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])

    encoder = keras.Model(inp, [z_mean, z_log_var, z], name="encoder")

    # ---------- Decoder ----------
    latent_inp = layers.Input(shape=(latent_dim,))
    x = latent_inp
    for units in decoder_units:
        x = layers.Dense(units, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)

    out = layers.Dense(input_dim, activation="sigmoid")(x)
    decoder = keras.Model(latent_inp, out, name="decoder")

    # ---------- VAE ----------
    _, _, z_sampled = encoder(inp)
    reconstruction = decoder(z_sampled)
    vae = keras.Model(inp, reconstruction, name="vae")

    # Loss: reconstruction (MSE) + beta * KL
    recon_loss = tf.reduce_mean(tf.square(inp - reconstruction), axis=1)
    kl_loss = -0.5 * tf.reduce_mean(
        1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1
    )
    vae.add_loss(tf.reduce_mean(recon_loss + beta * kl_loss))

    return vae, encoder, decoder

## 3. Configuration Grid

Each config varies architecture depth, latent size, and beta.

In [ ]:
CONFIGS = {
    "vae_shallow_z16_b1": {
        "encoder_units": [64, 32],
        "latent_dim": 16,
        "decoder_units": [32, 64],
        "dropout": 0.1,
        "beta": 1.0,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
    "vae_deep_z32_b1": {
        "encoder_units": [128, 64, 32],
        "latent_dim": 32,
        "decoder_units": [32, 64, 128],
        "dropout": 0.15,
        "beta": 1.0,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
    "vae_deep_z32_b05": {
        "encoder_units": [128, 64, 32],
        "latent_dim": 32,
        "decoder_units": [32, 64, 128],
        "dropout": 0.15,
        "beta": 0.5,
        "batch_size": 256,
        "lr": 1e-3,
        "epochs": 100,
        "patience": 10,
    },
    "vae_wide_z8_b2": {
        "encoder_units": [256, 128],
        "latent_dim": 8,
        "decoder_units": [128, 256],
        "dropout": 0.2,
        "beta": 2.0,
        "batch_size": 512,
        "lr": 5e-4,
        "epochs": 100,
        "patience": 10,
    },
}

## 4. Training Loop

In [ ]:
MODEL_FAMILY = "vae"

for cfg_name, cfg in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Training: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)

    vae, encoder, decoder = build_vae(
        input_dim=input_dim,
        encoder_units=cfg["encoder_units"],
        latent_dim=cfg["latent_dim"],
        decoder_units=cfg["decoder_units"],
        dropout=cfg["dropout"],
        beta=cfg["beta"],
    )

    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg["lr"]))
    vae.summary()

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss", patience=cfg["patience"], restore_best_weights=True
    )

    history = vae.fit(
        X_train, None,  # target=None because loss is added via add_loss
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        validation_split=0.1,
        shuffle=True,
        callbacks=[early_stop],
    )

    # --- Compute reconstruction error & threshold ---
    recon_train = vae.predict(X_train, batch_size=1024, verbose=0)
    train_errors = nu.reconstruction_mse(X_train, recon_train)
    threshold = nu.compute_threshold(train_errors, method="percentile", percentile=97)

    # --- Save everything ---
    vae.save(os.path.join(model_dir, f"{cfg_name}.keras"))
    encoder.save(os.path.join(model_dir, f"{cfg_name}_encoder.keras"))
    decoder.save(os.path.join(model_dir, f"{cfg_name}_decoder.keras"))

    nu.save_training_artifacts(
        model_dir, history, threshold,
        extra_meta={"config": cfg, "input_dim": input_dim, "model_type": "vae"},
    )
    nu.plot_error_distribution(
        train_errors, threshold,
        title=f"{cfg_name} — Train Error Distribution",
        save_path=os.path.join(model_dir, "error_dist.png"),
    )

    print(f"  Threshold (97th pctl): {threshold:.6f}")
    print(f"  Mean train MSE:        {train_errors.mean():.6f}")

print("\nAll VAE configs trained and saved.")

## 5. Quick Evaluation (All Models)

Evaluate every trained VAE config on the attack files and produce a combined comparison.

In [ ]:
all_eval_dfs = []

for cfg_name, cfg in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Evaluating: {cfg_name}")
    print(f"{'='*60}")

    model_dir = nu.get_model_dir(MODEL_FAMILY, cfg_name)
    model_path = os.path.join(model_dir, f"{cfg_name}.keras")
    threshold_path = os.path.join(model_dir, "threshold.json")

    # Load saved model and threshold
    saved_model = keras.models.load_model(model_path)
    with open(threshold_path) as f:
        saved_threshold = json.load(f)["threshold"]

    # Evaluate on attack files
    eval_df = nu.evaluate_on_attack_files(
        model=saved_model,
        scaler=scaler,
        feature_cols=feature_cols,
        threshold=saved_threshold,
        reshape_fn=None,  # VAE takes flat 2D input
    )
    eval_df.insert(0, "Model", cfg_name)

    # Save per-model quick_eval
    eval_df.to_csv(os.path.join(model_dir, "quick_eval.csv"), index=False)
    print(eval_df.to_string(index=False))

    all_eval_dfs.append(eval_df)

    del saved_model  # free memory

# ── Combined comparison across all models ──
combined_df = pd.concat(all_eval_dfs, ignore_index=True)

summary = combined_df.groupby("Model").agg({
    "Accuracy": "mean", "Precision": "mean", "Recall": "mean",
    "F1": "mean", "AUC": "mean",
}).reset_index().sort_values("F1", ascending=False)
summary.columns = ["Model", "Avg_Accuracy", "Avg_Precision", "Avg_Recall", "Avg_F1", "Avg_AUC"]

family_dir = os.path.join(nu.MODELS_ROOT, MODEL_FAMILY)
combined_df.to_csv(os.path.join(family_dir, "all_models_quick_eval.csv"), index=False)
summary.to_csv(os.path.join(family_dir, "model_comparison_summary.csv"), index=False)

print(f"\n{'='*60}")
print("  MODEL COMPARISON SUMMARY (sorted by Avg F1)")
print(f"{'='*60}")
print(summary.to_string(index=False))
print(f"\nCombined results saved → {os.path.join(family_dir, 'all_models_quick_eval.csv')}")
print(f"Summary saved → {os.path.join(family_dir, 'model_comparison_summary.csv')}")